# Silver Layer — Banking Transactions

## Overview
This notebook processes raw banking transaction data from
the Bronze layer into a clean, validated Silver Delta table.

**Source:** `banking_project.bronze.bronze_transactions`

**Target:** `banking_project.silver.silver_transactions`

**Key Operations:**
- Data type conversion
- Schema evolution handling
- Data quality validation
- Negative testing and quarantine
- Duplicate removal
- Delta table creation

## 1. Data Reading
Read raw transaction records from the Bronze Delta table.

In [0]:
bronze_table = "banking_project.bronze.bronze_transactions"
silver_table = "banking_project.silver.silver_transactions"

df_bronze = spark.table(bronze_table)

print("Bronze record count:", df_bronze.count())
df_bronze.printSchema()

## 2. Data Type Conversion

Convert raw transaction columns from the Bronze layer into appropriate data types for accurate processing and analysis.

**Transformations:**
- `transaction_amount`: STRING → DECIMAL(18,2) for financial calculations.
- `transaction_date`: STRING → DATE for date-based analysis.

**Purpose:** Ensure data consistency and prepare transaction records for downstream processing.

In [0]:
from pyspark.sql.functions import col, to_date

df_silver = (
    df_bronze
    .withColumn(
        "transaction_amount",
        col("transaction_amount").cast("decimal(18,2)")
    )
    .withColumn(
        "transaction_date",
        to_date(col("transaction_date"))
    )
)

df_silver.select(
    "transaction_id",
    "transaction_amount",
    "transaction_date"
).show(5)

df_silver.select(
    "transaction_amount",
    "transaction_date"
).printSchema()

## 3. Initial Data Quality Validation

Perform preliminary data quality checks to identify missing or invalid values in critical transaction fields.

**Validation Checks:**
- **Transaction ID:** Identify records with NULL transaction IDs.
- **Transaction Amount:** Identify NULL amounts, including values that could not be converted to DECIMAL.
- **Transaction Date:** Identify NULL dates, including values that could not be converted to DATE.

**Purpose:** Detect data quality issues early before further transformations and loading into the Silver layer.

**Note:** This step only identifies invalid records. Rejection and quarantine handling are implemented in later sections.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

df_silver.select(
    spark_sum(
        when(col("transaction_id").isNull(), 1).otherwise(0)
    ).alias("missing_transaction_ids"),

    spark_sum(
        when(col("transaction_amount").isNull(), 1).otherwise(0)
    ).alias("invalid_or_missing_amounts"),

    spark_sum(
        when(col("transaction_date").isNull(), 1).otherwise(0)
    ).alias("invalid_or_missing_dates")
).show()

## 4. Duplicate Transaction Detection

Identify duplicate transactions in the Silver DataFrame using `transaction_id` as the unique business key.

**Processing Steps:**
- Group transaction records by `transaction_id`.
- Count the occurrences of each transaction ID.
- Identify transaction IDs appearing more than once.

**Purpose:** Detect duplicate transactions that could lead to incorrect financial reporting or duplicate processing.

**Note:** This step only detects duplicates. Actual deduplication is performed later using `dropDuplicates()`.

In [0]:
df_duplicates = (
    df_silver
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
)

print("Duplicate transaction IDs:", df_duplicates.count())

df_duplicates.show(10)

## 5. Schema Evolution Handling

Handle new columns introduced in incoming transaction files using Auto Loader's rescued data mechanism.

**Scenario:**
- The initial transaction files did not contain `device_type`.
- A later batch introduced a new column called `device_type` with the value `Android`.
- Auto Loader was configured with `cloudFiles.schemaEvolutionMode = "rescue"`.
- The new column was preserved inside `_rescued_data` instead of being added directly to the Bronze table schema.

**Transformation:**
- Use `get_json_object()` to extract `device_type` from `_rescued_data`.
- Add `device_type` as a separate column in the Silver DataFrame.
- Records without this attribute retain a NULL value.

**Purpose:** Preserve newly introduced source attributes without losing transaction records during schema changes.

In [0]:
from pyspark.sql.functions import get_json_object, col

df_silver = df_silver.withColumn(
    "device_type",
    get_json_object(col("_rescued_data"), "$.device_type")
)

df_silver.select(
    "transaction_id",
    "device_type"
).filter(
    col("device_type").isNotNull()
).show(15)

## 6. Business Rule Validation

Perform business-level data quality checks to ensure transaction records satisfy essential banking data requirements.

**Validation Rules:**
- **Transaction Amount:** Identify transactions with zero or negative amounts.
- **Customer ID:** Identify records with NULL or empty customer IDs.
- **Transaction ID:** Identify records with NULL or empty transaction IDs.

**Business Assumption:** Transaction amounts must be positive because transaction direction is represented separately.

**Purpose:** Identify invalid transactions that could affect downstream financial reporting and analytics.

**Note:** This step identifies rule violations. Invalid records are separated from valid records during the final data quality validation stage.

In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.sum(
        F.when(F.col("transaction_amount") <= 0, 1).otherwise(0)
    ).alias("non_positive_amounts"),

    F.sum(
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_customer_ids"),

    F.sum(
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            1
        ).otherwise(0)
    ).alias("missing_or_empty_transaction_ids")
).show()

## 7. Negative Testing — Creating Invalid Transactions

Create deliberately invalid transaction records to test the effectiveness of our data quality validation rules.

**Test Scenarios:**
- **TEST001:** Negative transaction amount (`-500.00`).
- **TEST002:** Missing customer ID (`NULL`).
- **Test record 3:** Empty transaction ID.

**Implementation:**
- Select three sample records from the Silver DataFrame.
- Modify selected fields to introduce data quality issues.
- Store these records in a separate test DataFrame (`df_invalid_test`).

**Purpose:** Verify that invalid banking transactions can be identified and quarantined before entering the Silver Delta table.

**Note:** These are synthetic test records created in memory. They are not inserted into the Bronze or Silver Delta tables.

In [0]:

from pyspark.sql import functions as F

# Select 3 sample transactions
sample_rows = (
    df_silver
    .orderBy("transaction_id")
    .limit(3)
    .collect()
)

# Create invalid test records
df_invalid_test = spark.createDataFrame(
    sample_rows,
    schema=df_silver.schema
)

# Give each record a deliberate data quality problem
df_invalid_test = (
    df_invalid_test
    .withColumn(
        "transaction_id",
        F.when(F.col("transaction_id") == sample_rows[0]["transaction_id"], "TEST001")
         .when(F.col("transaction_id") == sample_rows[1]["transaction_id"], "TEST002")
         .otherwise("")
    )
    .withColumn(
        "transaction_amount",
        F.when(F.col("transaction_id") == "TEST001", F.lit(-500).cast("decimal(18,2)"))
         .otherwise(F.col("transaction_amount"))
    )
    .withColumn(
        "customer_id",
        F.when(F.col("transaction_id") == "TEST002", F.lit(None).cast("string"))
         .otherwise(F.col("customer_id"))
    )
)

df_invalid_test.select(
    "transaction_id",
    "transaction_amount",
    "customer_id"
).show(truncate=False)


## 8. Quarantine Validation — Separating Valid and Invalid Records

Validate deliberately corrupted test transactions and separate them based on predefined data quality rules.

**Validation Rules:**
- **INVALID_AMOUNT:** Transaction amount is NULL, zero, or negative.
- **MISSING_CUSTOMER_ID:** Customer ID is NULL or empty.
- **MISSING_TRANSACTION_ID:** Transaction ID is NULL or empty.

**Processing Steps:**
1. Apply validation rules using PySpark `when()` conditions.
2. Generate a `rejection_reason` for each invalid record.
3. Store rejected records in `df_quarantine`.
4. Store accepted records in `df_valid`.
5. Verify the number of valid and quarantined records.

**Test Results:**
- Total test records: **3**
- Quarantined records: **3**
- Valid records: **0**

**Purpose:** Demonstrate that invalid transactions are identified and excluded from accepted records.

**Note:** Quarantine is tested in memory. A permanent quarantine Delta table has not yet been created.

In [0]:

from pyspark.sql import functions as F

# Add rejection reasons to invalid records
df_validated = df_invalid_test.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        )
    )
)

# Separate invalid records
df_quarantine = df_validated.filter(
    F.col("rejection_reason") != ""
)

# Separate valid records
df_valid = df_validated.filter(
    F.col("rejection_reason") == ""
)

print("Total test records:", df_validated.count())
print("Quarantined records:", df_quarantine.count())
print("Valid records:", df_valid.count())

df_quarantine.select(
    "transaction_id",
    "transaction_amount",
    "customer_id",
    "rejection_reason"
).show(truncate=False)


## 9. Data Quality Validation — Actual Transaction Records

Apply data quality rules to the actual banking transaction dataset.

**Validation Rules:**
- **INVALID_AMOUNT:** Transaction amount is NULL, zero, or negative.
- **MISSING_CUSTOMER_ID:** Customer ID is NULL or empty.
- **MISSING_TRANSACTION_ID:** Transaction ID is NULL or empty.
- **INVALID_TRANSACTION_DATE:** Transaction date is NULL.

**Processing Steps:**
1. Apply validation rules to the Silver DataFrame.
2. Generate rejection reasons for invalid transactions.
3. Separate accepted records into `df_valid_all`.
4. Separate rejected records into `df_quarantine_all`.
5. Verify the record counts.

**Validation Results:**
- Total Bronze records: **10,110**
- Valid records: **10,110**
- Quarantined records: **0**

**Purpose:** Verify that actual banking transactions satisfy the initial data quality requirements before further transformations.

**Note:** This is a preliminary validation. Additional numeric checks and final validation are performed later in the notebook.

In [0]:
from pyspark.sql import functions as F

df_validated_all = df_silver.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",
        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        ),
        F.when(
            F.col("transaction_date").isNull(),
            "INVALID_TRANSACTION_DATE"
        )
    )
)

df_valid_all = df_validated_all.filter(
    F.col("rejection_reason") == ""
)

df_quarantine_all = df_validated_all.filter(
    F.col("rejection_reason") != ""
)

print("Bronze records:", df_bronze.count())
print("Valid records:", df_valid_all.count())
print("Quarantined records:", df_quarantine_all.count())

In [0]:
## 10. Additional Data Type Conversion

Convert additional banking transaction attributes into appropriate data types for accurate calculations and downstream analysis.

**Transformations:**
- `account_balance`: STRING → DECIMAL(18,2)
- `emi_amount`: STRING → DECIMAL(18,2)
- `credit_score`: STRING → INTEGER
- `transaction_hour`: STRING → INTEGER

**Implementation:**
- Use PySpark `withColumn()` to transform the existing columns.
- Use `try_cast()` to safely convert values into the required data types.
- If a value cannot be converted, `try_cast()` returns NULL instead of raising a conversion error.

**Purpose:** Ensure numeric attributes have consistent data types and identify malformed values for subsequent data quality validation.

**Note:** Type conversion alone does not guarantee data quality. NULL values and invalid numeric ranges are checked in the next section.

In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_silver
    .withColumn(
        "account_balance",
        F.expr("try_cast(account_balance AS DECIMAL(18,2))")
    )
    .withColumn(
        "emi_amount",
        F.expr("try_cast(emi_amount AS DECIMAL(18,2))")
    )
    .withColumn(
        "credit_score",
        F.expr("try_cast(credit_score AS INT)")
    )
    .withColumn(
        "transaction_hour",
        F.expr("try_cast(transaction_hour AS INT)")
    )
)

df_silver.select(
    "account_balance",
    "emi_amount",
    "credit_score",
    "transaction_hour"
).printSchema()

df_silver.select(
    "account_balance",
    "emi_amount",
    "credit_score",
    "transaction_hour"
).show(5)

## 11. Numeric Data Quality Validation

Validate the converted numeric columns to identify missing, malformed, or out-of-range values.

**Validation Rules:**
- **Account Balance:** Identify NULL values resulting from missing data or failed type conversions.
- **EMI Amount:** Identify NULL values resulting from missing data or failed type conversions.
- **Credit Score:** Identify non-NULL values outside the expected range of 300–900.
- **Transaction Hour:** Identify NULL values or values outside the valid range of 0–23.

**Implementation:**
- Use PySpark `when()` conditions to identify invalid values.
- Use `sum()` to count records violating each validation rule.
- Display the results for data quality monitoring.

**Validation Results:**
- Invalid account balances: **0**
- Invalid EMI amounts: **0**
- Invalid credit scores: **0**
- Invalid transaction hours: **0**

**Purpose:** Verify numeric data quality before deduplication and final Silver processing.

**Note:** NULL credit scores are allowed by this particular validation rule. Additional business requirements may require stricter checks.

In [0]:
from pyspark.sql import functions as F

df_silver.select(
    F.sum(
        F.when(F.col("account_balance").isNull(), 1).otherwise(0)
    ).alias("invalid_account_balance"),

    F.sum(
        F.when(F.col("emi_amount").isNull(), 1).otherwise(0)
    ).alias("invalid_emi_amount"),

    F.sum(
        F.when(
            F.col("credit_score").isNotNull() &
            ~F.col("credit_score").between(300, 900),
            1
        ).otherwise(0)
    ).alias("invalid_credit_score"),

    F.sum(
        F.when(
            F.col("transaction_hour").isNull() |
            ~F.col("transaction_hour").between(0, 23),
            1
        ).otherwise(0)
    ).alias("invalid_transaction_hour")
).show()

## 12. Data Deduplication

Remove duplicate transaction records using `transaction_id` as the unique business key.

**Processing Steps:**
- Count the total records before deduplication.
- Use PySpark `dropDuplicates(["transaction_id"])` to retain one record per transaction ID.
- Count the records after deduplication.
- Calculate the number of duplicates removed.

**Implementation:**
- Input DataFrame: `df_silver`
- Output DataFrame: `df_silver_dedup`

**Results:**
- Records before deduplication: **10,110**
- Records after deduplication: **10,110**
- Duplicates removed: **0**

**Purpose:** Prevent duplicate transactions from affecting financial reporting and downstream analytics.

**Note:** `dropDuplicates()` does not guarantee which record is retained when duplicate transaction IDs contain different values. For CDC or incremental updates, a timestamp-based deduplication strategy is more appropriate.

In [0]:
# Count records before deduplication
before_count = df_silver.count()

# Keep one record per transaction ID
df_silver_dedup = df_silver.dropDuplicates(
    ["transaction_id"]
)

# Count records after deduplication
after_count = df_silver_dedup.count()

print("Before deduplication:", before_count)
print("After deduplication:", after_count)
print("Duplicates removed:", before_count - after_count)

## 13. Final Data Quality Validation and Quarantine Separation

Apply the final data quality rules to the deduplicated Silver DataFrame before writing records to the Silver Delta table.

**Validation Rules:**
- **Transaction ID:** Must not be NULL or empty.
- **Customer ID:** Must not be NULL or empty.
- **Transaction Amount:** Must be greater than zero.
- **Transaction Date:** Must not be NULL.
- **Account Balance:** Must not be NULL.
- **EMI Amount:** Must not be NULL.
- **Credit Score:** If present, must be between 300 and 900.
- **Transaction Hour:** Must be between 0 and 23.

**Processing Steps:**
1. Apply validation rules using PySpark `when()` conditions.
2. Generate a `rejection_reason` for records violating one or more rules.
3. Separate valid records into `df_final_valid`.
4. Separate invalid records into `df_final_quarantine`.
5. Count accepted and rejected transactions.

**Validation Results:**
- Total records: **10,110**
- Valid records: **10,110**
- Quarantined records: **0**

**Purpose:** Ensure that only transactions satisfying the defined data quality rules are written to the Silver Delta table.

**Note:** Rejected records are currently separated into an in-memory DataFrame. Persistent quarantine storage has not yet been implemented.

In [0]:
from pyspark.sql import functions as F

df_final_checked = df_silver_dedup.withColumn(
    "rejection_reason",
    F.concat_ws(
        "; ",

        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "MISSING_TRANSACTION_ID"
        ),

        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "MISSING_CUSTOMER_ID"
        ),

        F.when(
            F.col("transaction_amount").isNull() |
            (F.col("transaction_amount") <= 0),
            "INVALID_AMOUNT"
        ),

        F.when(
            F.col("transaction_date").isNull(),
            "INVALID_DATE"
        ),

        F.when(
            F.col("account_balance").isNull(),
            "INVALID_ACCOUNT_BALANCE"
        ),

        F.when(
            F.col("emi_amount").isNull(),
            "INVALID_EMI_AMOUNT"
        ),

        F.when(
            F.col("credit_score").isNotNull() &
            ~F.col("credit_score").between(300, 900),
            "INVALID_CREDIT_SCORE"
        ),

        F.when(
            F.col("transaction_hour").isNull() |
            ~F.col("transaction_hour").between(0, 23),
            "INVALID_TRANSACTION_HOUR"
        )
    )
)

# Separate accepted and rejected transactions
df_final_valid = df_final_checked.filter(
    F.col("rejection_reason") == ""
).drop("rejection_reason")

df_final_quarantine = df_final_checked.filter(
    F.col("rejection_reason") != ""
)

print("Total records:", df_final_checked.count())
print("Valid records:", df_final_valid.count())
print("Quarantined records:", df_final_quarantine.count())

## 14. Data Writing — Silver Delta Table

Write cleaned, validated, and deduplicated banking transactions into a Delta table governed by Unity Catalog.

**Target Table:** `banking_project.silver.silver_transactions`

**Processing Steps:**
- Use `df_final_valid` containing transactions that passed all data quality checks.
- Write records using the Delta Lake format.
- Use `saveAsTable()` to create the Silver table if it does not already exist.
- Check whether the target table exists before performing the initial load.

**Write Strategy:**
- **Format:** Delta
- **Mode:** Append (for initial table creation)
- **Catalog:** `banking_project`
- **Schema:** `silver`
- **Table:** `silver_transactions`

**Purpose:** Persist validated transaction data for downstream analytics and Gold-layer processing.

**Note:** The current implementation uses a table-existence check to prevent repeated initial inserts. Incremental inserts and updates using `MERGE INTO` will be implemented in a later stage.

In [0]:
silver_table = "banking_project.silver.silver_transactions"

if not spark.catalog.tableExists(silver_table):

    (
        df_final_valid.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("Silver Delta table created successfully!")

else:
    print("Silver table already exists. No records written.")

## 15. Post-Load Validation — Silver Delta Table

Verify that the cleaned and validated banking transactions have been successfully persisted in the Silver Delta table.

**Target Table:** `banking_project.silver.silver_transactions`

**Validation Steps:**
- Query the Silver Delta table using Databricks SQL.
- Count the total number of records stored in the table.
- Confirm that the record count matches the expected number of valid transactions.

**Validation Results:**
- Bronze transaction records: **10,110**
- Valid transactions after data quality checks: **10,110**
- Silver Delta table records: **10,110**
- Quarantined records: **0**

**Purpose:** Confirm that the initial Silver data load completed successfully and that the expected number of validated transactions is available for downstream Gold-layer processing.

**Note:** Record-count validation confirms the expected volume, but additional checks such as key uniqueness and data reconciliation are recommended for production pipelines.

In [0]:
%sql
SELECT COUNT(*)
FROM banking_project.silver.silver_transactions;